# 📊 US Superstore — Data Analysis for Marketing Strategy

End-to-end analysis covering area, customer, and product dimensions with Pareto application.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

# Styling
plt.rcParams.update({
    'figure.facecolor': '#FFFFFF', 'axes.facecolor': '#F1F5F9',
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})
BLUE, GREEN, AMBER, RED, PURPLE = '#2563EB', '#10B981', '#F59E0B', '#EF4444', '#8B5CF6'

def money(x, pos=None):
    if x >= 1_000_000: return f'${{x/1_000_000:.1f}}M'
    if x >= 1_000:     return f'${{x/1_000:.0f}}K'
    return f'${{x:.0f}}'


## 1. Load & Preprocess

In [ ]:
df = pd.read_excel('US_Superstore_data.xls', engine='xlrd')
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Year'] = df['Order Date'].dt.year

print(f"Shape: {df.shape}")
print(f"Date range: {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()


## 2. Which states have the most sales?

In [ ]:
state_grp = df.groupby('State').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
top20 = state_grp.nlargest(20, 'Sales')

fig, ax = plt.subplots(figsize=(10, 7))
colors = [AMBER if s == 'California' else BLUE for s in top20['State']]
ax.barh(top20['State'], top20['Sales'], color=colors, height=0.7)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(money))
ax.set_title('Top 20 States by Sales', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.show()
print("Top 5 states by sales:")
print(top20[['State','Sales']].head().to_string(index=False))


## 3. New York vs California

In [ ]:
ny_ca = df[df['State'].isin(['New York','California'])]
compare = ny_ca.groupby('State').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
compare['Margin%'] = compare['Profit'] / compare['Sales'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, title in zip(axes, ['Sales','Profit'], ['Total Sales','Total Profit']):
    bars = ax.bar(compare['State'], compare[col], color=[BLUE, AMBER], width=0.5)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(money))
    ax.set_title(f'{title}: NY vs CA', fontweight='bold')
    for b, v in zip(bars, compare[col]):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+1000, money(v),
                ha='center', fontweight='bold')
plt.tight_layout(); plt.show()
print(compare[['State','Sales','Profit','Margin%']].to_string(index=False))


## 4. Outstanding Customer in New York

In [ ]:
ny_customers = df[df['State']=='New York'].groupby('Customer Name').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique')
).reset_index().nlargest(10,'Sales')

fig, ax = plt.subplots(figsize=(10,5))
colors = [AMBER if i==0 else BLUE for i in range(len(ny_customers))]
ax.barh(ny_customers['Customer Name'], ny_customers['Sales'], color=colors, height=0.6)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(money))
ax.set_title('Top 10 NY Customers by Sales  ★ = Outstanding', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.show()
print(f"Outstanding NY customer: {ny_customers.iloc[0]['Customer Name']} — {money(ny_customers.iloc[0]['Sales'])}")


## 5. Profitability by State

In [ ]:
state_sorted = state_grp.sort_values('Profit', ascending=True)
bar_colors = [GREEN if p >= 0 else RED for p in state_sorted['Profit']]

fig, ax = plt.subplots(figsize=(10,12))
ax.barh(state_sorted['State'], state_sorted['Profit'], color=bar_colors, height=0.7)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(money))
ax.set_title('Profit by State', fontsize=14, fontweight='bold')
ax.invert_yaxis(); ax.axvline(0, color='gray', lw=1.2)
ax.tick_params(axis='y', labelsize=8)
handles = [mpatches.Patch(color=GREEN, label='Profitable'), mpatches.Patch(color=RED, label='Loss')]
ax.legend(handles=handles)
plt.tight_layout(); plt.show()
loss_states = state_grp[state_grp.Profit < 0]['State'].tolist()
print(f"Loss-making states ({len(loss_states)}): {loss_states}")


## 6. Pareto Principle — Customers & Profit

In [ ]:
cust_profit = df.groupby('Customer Name')['Profit'].sum().sort_values(ascending=False).reset_index()
cust_profit['CumProfit%'] = cust_profit['Profit'].cumsum() / cust_profit['Profit'].sum() * 100
cust_profit['CustPct'] = np.arange(1, len(cust_profit)+1) / len(cust_profit) * 100

n20 = int(len(cust_profit)*0.2)
p20 = cust_profit.iloc[n20-1]['CumProfit%']

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(cust_profit['CustPct'], cust_profit['CumProfit%'], color=BLUE, lw=2.5)
ax.fill_between(cust_profit['CustPct'], cust_profit['CumProfit%'], alpha=0.12, color=BLUE)
ax.axvline(20, color=RED, ls='--', lw=1.5, label='Top 20% customers')
ax.axhline(80, color=AMBER, ls='--', lw=1.5, label='80% profit threshold')
ax.plot(20, p20, 'o', color=RED, ms=9)
ax.annotate(f'Top 20% → {p20:.1f}% profit', xy=(20,p20), xytext=(35, p20-12),
            arrowprops=dict(arrowstyle='->'), fontsize=9,
            bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set(xlabel='Customers (cumulative %)', ylabel='Cumulative Profit (%)',
       title='Pareto: Customers vs Profit', xlim=(0,100), ylim=(0,100))
ax.legend(); plt.tight_layout(); plt.show()
print(f"Top 20% of customers → {p20:.1f}% of total profit  {'✅ Pareto holds!' if p20>=80 else '⚠️ Close but not 80%'}")


## 7. Top 20 Cities by Sales & Profit

In [ ]:
city_grp = df.groupby('City').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
top20_s = city_grp.nlargest(20,'Sales')
top20_p = city_grp.nlargest(20,'Profit')
top20_s['Margin%'] = top20_s['Profit']/top20_s['Sales']*100

fig, axes = plt.subplots(1, 3, figsize=(20,7))
axes[0].barh(top20_s['City'], top20_s['Sales'], color=BLUE, height=0.7)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(money))
axes[0].set_title('Top 20 Cities — Sales', fontweight='bold'); axes[0].invert_yaxis()

bc = [GREEN if p>=0 else RED for p in top20_p['Profit']]
axes[1].barh(top20_p['City'], top20_p['Profit'], color=bc, height=0.7)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(money))
axes[1].set_title('Top 20 Cities — Profit', fontweight='bold'); axes[1].invert_yaxis()

cm = [GREEN if m>=0 else RED for m in top20_s['Margin%']]
axes[2].scatter(top20_s['Sales'], top20_s['Margin%'], c=cm, s=100)
for _, r in top20_s.iterrows():
    axes[2].annotate(r['City'], (r['Sales'], r['Margin%']), textcoords='offset points',
                     xytext=(4,3), fontsize=7)
axes[2].axhline(0, color='gray', lw=1); axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(money))
axes[2].set(xlabel='Sales', ylabel='Margin %', title='Sales vs Margin (top 20 cities)')
plt.tight_layout(); plt.show()


## 8. Top 20 Customers by Sales

In [ ]:
cust_sales = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False).reset_index()
top20_cust = cust_sales.head(20)

fig, ax = plt.subplots(figsize=(10,7))
colors = [AMBER if i<3 else BLUE for i in range(20)]
ax.barh(top20_cust['Customer Name'], top20_cust['Sales'], color=colors, height=0.7)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(money))
ax.set_title('Top 20 Customers by Sales  (Gold = Top 3)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.show()
print("Top 5 customers:")
print(top20_cust.head().to_string(index=False))


## 9. Cumulative Sales Curve by Customers (Pareto Check)

In [ ]:
cust_sales['CumSales%'] = cust_sales['Sales'].cumsum() / cust_sales['Sales'].sum() * 100
cust_sales['CustPct'] = np.arange(1, len(cust_sales)+1) / len(cust_sales) * 100
s20 = cust_sales.iloc[int(len(cust_sales)*0.2)-1]['CumSales%']

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(cust_sales['CustPct'], cust_sales['CumSales%'], color=GREEN, lw=2.5, label='Cumulative Sales')
ax.plot([0,100],[0,100], color='gray', ls=':', lw=1.2, label='Perfect equality')
ax.fill_between(cust_sales['CustPct'], cust_sales['CumSales%'], alpha=0.12, color=GREEN)
ax.axvline(20, color=RED, ls='--', lw=1.5)
ax.axhline(80, color=AMBER, ls='--', lw=1.5)
ax.annotate(f'Top 20% → {s20:.1f}% Sales', xy=(20,s20), xytext=(35,s20-12),
            arrowprops=dict(arrowstyle='->'), fontsize=9,
            bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set(xlabel='Customers (cumulative %)', ylabel='Cumulative Sales (%)',
       title='Cumulative Sales Curve (Pareto Check)', xlim=(0,100), ylim=(0,100))
ax.legend(); plt.tight_layout(); plt.show()
print(f"Top 20% customers → {s20:.1f}% of total sales  {'⚠️ Pareto does NOT hold for sales' if s20<80 else '✅ Pareto holds!'}")


## 10. Marketing Strategy Recommendations

### 🎯 Priority States
- **California & New York** are #1 and #2 by sales and maintain strong profit margins (~16%). Invest in retention and upselling.
- **Washington, Michigan, Virginia** are high-sales AND profitable — expand presence.
- **Texas, Illinois, Ohio, Pennsylvania** generate high sales but are **loss-making** — review discount policies before further spend.

### 🏙️ Priority Cities
- **New York City, Los Angeles, Seattle, San Francisco** are top by both sales and profit.
- Avoid heavy marketing in cities where margin is negative despite high sales (e.g. Philadelphia, Houston).

### 👥 Customer Strategy
- **Pareto holds for Profit** (top 20% → 81% of profit): build a VIP loyalty program.
- **Pareto does NOT hold for Sales** (top 20% → ~48%): sales are more evenly distributed, so broad acquisition matters too.
- **Tom Ashbrook** is the outstanding NY customer — worth dedicated account management.

### ⚠️ Red Flags
- 10 states are currently unprofitable — likely due to excessive discounting. Audit discount strategy in Texas, Florida, Ohio, Illinois.
